Create a Tokenizer

Tokenizers convert text → numbers.

In [1]:
from tokenizers import ByteLevelBPETokenizer
import os
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=['C:/shiva/coding/python_projects/dear_comrade_data_science_zero_to_hero/data.txt'],
    vocab_size=5000,
    min_frequency=2,
)
os.makedirs("./dha_llm_tokenizer", exist_ok=True)
tokenizer.save_model("./dha_llm_tokenizer")

['./dha_llm_tokenizer\\vocab.json', './dha_llm_tokenizer\\merges.txt']

Build a Small Transformer Model
Now we create a mini GPT-style mode

In [2]:
import torch
import torch.nn as nn

class DHA(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, 256)
        self.transformer  = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=256, nhead=4),
            num_layers=4
        )
        self.fc = nn.Linear(256, vocab_size)
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = self.fc(x)
        return x

Load Dataset for Training

Convert text → tokens.

In [3]:
from transformers import AutoTokenizer
from tokenizers import ByteLevelBPETokenizer

tokenizer_new = ByteLevelBPETokenizer(
    "./dha_llm_tokenizer/vocab.json",
    "./dha_llm_tokenizer/merges.txt"
)
# tokenizer_new = AutoTokenizer.from_pretrained("./dha_llm_tokenizer")
text = open('C:/shiva/coding/python_projects/dear_comrade_data_science_zero_to_hero/data.txt').read()
tokens = tokenizer_new.encode(text)
tokens = torch.tensor(tokens)



C:\shiva\coding\python_projects\dear_comrade_data_science_zero_to_hero\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Could not infer dtype of tokenizers.Encoding

Training Loop

This is where the model learns.


In [ ]:
model = DHA(vocab_size=5000)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = torch.nn.CrossEntropyLoss()
for epoch in range(10):
    optimizer.zero_grad()
    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    output = model(input_tokens)
    loss = loss_fn(output.view(-1, 5000), target_tokens.view(-1))
    loss.backward()
    optimizer.step()
    print("Loss:", loss.item())

Generate Text

After training:

In [ ]:
prompt = "Artificial intelligence"

tokens = tokenizer.encode(prompt)

tokens = torch.tensor(tokens)

output = model(tokens)

next_token = torch.argmax(output[-1])

print(tokenizer.decode([next_token]))